[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/klar74/WS2026_lecture/blob/main/Vorlesung_21/iris_keras_neural_network.ipynb)

# Neuronale Netze mit TensorFlow/Keras: Iris-Datensatz

Wir bauen ein Netz mit mehreren versteckten Schichten und betrachten:
- getrennte Trainings-, Validierungs- und Testdaten,
- Lernkurven für Training und Validierung,
- Early Stopping und die Anpassung der Lernrate,
- Modellwahrscheinlichkeiten sowie Speichern und Laden.

**Voraussetzung:** TensorFlow ist erforderlich und in der
[environment.yml](../environment.yml) enthalten. GPU-Nutzung hängt von Hardware
und Installation ab; für dieses kleine Beispiel genügt die CPU.

## 1. Import Libraries und Setup

Zuerst laden wir TensorFlow/Keras und alle anderen benötigten Bibliotheken.

In [ ]:
# TensorFlow und Keras
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers
from tensorflow.keras.utils import to_categorical

# Grundlegende Bibliotheken
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

# Scikit-learn für Daten und Metriken
from sklearn.datasets import load_iris
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix

# Plotting Setup
plt.style.use('default')
sns.set_palette("husl")

# Zufallszahlen und TensorFlow-Operationen möglichst reproduzierbar machen
keras.utils.set_random_seed(42)
tf.config.experimental.enable_op_determinism()

print("🚀 TensorFlow/Keras Setup:")
print(f"TensorFlow Version: {tf.__version__}")
print(f"Keras Version: {keras.__version__}")
print(f"GPU verfügbar: {len(tf.config.list_physical_devices('GPU')) > 0}")
print("Zufallsstart festgelegt. Andere Software- oder Hardwareversionen können Ergebnisse verändern.")
print("✅ Alle Bibliotheken erfolgreich importiert!")

## 2. Daten laden und aufteilen

Wir trennen zuerst Training, Validierung und Test. Nur die Trainingsdaten bestimmen
die Skalierung. Die Validierung hilft beim Steuern des Trainings; der Test bleibt
bis zur abschließenden Bewertung unbenutzt.

Für die verwendete Verlustfunktion stellen wir die Labels als One-Hot-Vektoren dar:
zum Beispiel Klasse 1 als `[0, 1, 0]`. Keras kann mit einer anderen Verlustfunktion
auch direkt mit Klassennummern arbeiten.

In [ ]:
iris = load_iris()
X, y = iris.data, iris.target

X_trainval, X_test, y_trainval, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)
X_train, X_val, y_train, y_val = train_test_split(
    X_trainval, y_trainval, test_size=0.2, random_state=42, stratify=y_trainval
)

print(f"Training: {len(X_train)}, Validierung: {len(X_val)}, Test: {len(X_test)} Blüten")
print(f"Klassen: {iris.target_names}")

scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_val_scaled = scaler.transform(X_val)
X_test_scaled = scaler.transform(X_test)

y_train_onehot = to_categorical(y_train, num_classes=3)
y_val_onehot = to_categorical(y_val, num_classes=3)
y_test_onehot = to_categorical(y_test, num_classes=3)

print(f"Training nach Skalierung: Mittelwerte {X_train_scaled.mean(axis=0).round(3)}")
print(f"Standardabweichungen: {X_train_scaled.std(axis=0).round(3)}")
print(f"One-Hot-Beispiel: {y_train[0]} → {y_train_onehot[0]}")
print(f"Klassenverteilung Training: {np.bincount(y_train)}")
print(f"Klassenverteilung Validierung: {np.bincount(y_val)}")

## 3. Das Keras-Modell

Wir verwenden drei versteckte Schichten. ReLU lässt positive Werte durch und setzt
negative auf null. Softmax liefert drei Modellwahrscheinlichkeiten, die zusammen 1 ergeben.

Dropout setzt beim Training zufällig einige Ausgaben auf null und kann Überanpassung
verringern. Batch Normalization normalisiert Zwischenausgaben und kann das Training
stabilisieren. Beides sind Möglichkeiten, keine Garantie für bessere Ergebnisse.

In [ ]:
# Modell mit Sequential API erstellen
model = keras.Sequential([
    # Input Layer - explizit definiert
    layers.Input(shape=(4,), name='features'),

    # Hidden Layer 1 mit Batch Normalization
    layers.Dense(64, activation='relu', name='hidden1'),
    layers.BatchNormalization(name='bn1'),
    layers.Dropout(0.3, name='dropout1'),

    # Hidden Layer 2
    layers.Dense(32, activation='relu', name='hidden2'),
    layers.BatchNormalization(name='bn2'),
    layers.Dropout(0.2, name='dropout2'),

    # Hidden Layer 3 (optional - experimentiere!)
    layers.Dense(16, activation='relu', name='hidden3'),

    # Output Layer - Softmax für Multi-Class
    layers.Dense(3, activation='softmax', name='output')
], name='IrisClassifier')

# Modell-Architektur anzeigen
print("🧠 Neuronales Netz Architektur:")
model.summary()

# Modell kompilieren
model.compile(
    optimizer=keras.optimizers.Adam(learning_rate=0.001),
    loss='categorical_crossentropy',  # Für One-Hot Labels
    metrics=['accuracy']
)

print("\n⚙️ Kompilierung abgeschlossen:")
print(f"Optimizer: Adam (lr=0.001)")
print(f"Loss: Categorical Crossentropy")
print(f"Metriken: Accuracy + Loss")

# Parameter zählen
total_params = model.count_params()
print(f"\n📊 Model Statistiken:")
trainable_params = sum(int(np.prod(weight.shape)) for weight in model.trainable_weights)
print(f"Parameter insgesamt: {total_params:,}")
print(f"Davon trainierbar: {trainable_params:,}")
print(f"Nicht trainierbar: {total_params - trainable_params:,}")

## 4. Training mit Validierungsdaten

**Early Stopping** beendet das Training, wenn der Validierungsfehler 20 Epochen lang
nicht ausreichend besser wird. Anschließend werden die Gewichte mit dem kleinsten
Validierungsfehler wiederhergestellt. Es erkennt Overfitting nicht automatisch.

**ReduceLROnPlateau** verkleinert die Lernrate, wenn sich der Validierungsfehler längere
Zeit nicht verbessert. Eine Epoche ist ein Durchlauf durch die Trainingsdaten.

In [ ]:
# Callbacks für smartes Training
callbacks = [
    # Stoppt bei ausbleibender Verbesserung des Validierungsfehlers
    keras.callbacks.EarlyStopping(
        monitor='val_loss',
        patience=20,
        restore_best_weights=True,
        verbose=1
    ),

    # Learning Rate Scheduler
    keras.callbacks.ReduceLROnPlateau(
        monitor='val_loss',
        factor=0.5,
        patience=10,
        min_lr=0.0001,
        verbose=1
    )
]

print("🎯 Callbacks konfiguriert:")
print("✅ Early Stopping (patience=20)")
print("✅ Learning Rate Reduction")

# Training mit separat vorbereiteten Validierungsdaten
print("\n🚀 Training gestartet...")
history = model.fit(
    X_train_scaled, y_train_onehot,
    validation_data=(X_val_scaled, y_val_onehot),
    epochs=200,
    batch_size=16,
    callbacks=callbacks,
    verbose=0,  # Kompakte Ausgabe
    shuffle=True,  # Trainingsreihenfolge pro Epoche mischen
    validation_batch_size=16  # Validation Batch Size für Konsistenz
)

print(f"\n✅ Training beendet nach {len(history.history['loss'])} Epochen")

## 5. Lernkurven verstehen

Wir zeigen Fehler und Accuracy während des Trainings und auf der Validierung.
Steigt der Validierungsfehler, während der Trainingsfehler sinkt, kann das auf
Überanpassung hinweisen. Ein einzelner Abstand beweist aber weder Overfitting
noch gute Generalisierung.

Dropout ist nur im Training aktiv; auch Batch Normalization arbeitet bei der
Validierung anders. Deshalb sind beide Kurven nicht unter exakt gleichen Bedingungen
entstanden. Die letzte Epoche in der Kurve ist außerdem nicht unbedingt die Epoche
der von Early Stopping wiederhergestellten Gewichte.

In [ ]:
# Training History extrahieren
train_loss = history.history['loss']
val_loss = history.history['val_loss']
train_acc = history.history['accuracy']
val_acc = history.history['val_accuracy']

epochs = range(1, len(train_loss) + 1)

# Subplot für Loss und Accuracy
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(15, 6))

# Loss Plot
ax1.plot(epochs, train_loss, 'b-', label='Training Loss', linewidth=2)
ax1.plot(epochs, val_loss, 'r-', label='Validation Loss', linewidth=2)
ax1.set_title('Training & Validation Loss', fontsize=14, fontweight='bold')
ax1.set_xlabel('Epochs')
ax1.set_ylabel('Loss')
ax1.legend()
ax1.grid(True, alpha=0.3)

# Accuracy Plot
ax2.plot(epochs, train_acc, 'b-', label='Training Accuracy', linewidth=2)
ax2.plot(epochs, val_acc, 'r-', label='Validation Accuracy', linewidth=2)
ax2.set_title('Training & Validation Accuracy', fontsize=14, fontweight='bold')
ax2.set_xlabel('Epochs')
ax2.set_ylabel('Accuracy')
ax2.legend()
ax2.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

# Training-Statistiken
min_val_loss = min(val_loss)
max_val_acc = max(val_acc)
final_train_loss = train_loss[-1]
final_val_loss = val_loss[-1]

print("📊 Training-Ergebnisse:")
print(f"Beste Validation Loss: {min_val_loss:.4f}")
print(f"Beste Validation Accuracy: {max_val_acc:.4f} ({max_val_acc*100:.2f}%)")
print(f"Final Training Loss: {final_train_loss:.4f}")
print(f"Final Validation Loss: {final_val_loss:.4f}")

best_epoch = int(np.argmin(val_loss)) + 1
print(f"Epoche mit kleinstem Validierungsfehler: {best_epoch}")
restored_val_loss, restored_val_accuracy = model.evaluate(X_val_scaled, y_val_onehot, verbose=0)
print(f"Wiederhergestelltes Modell: Validation Loss={restored_val_loss:.4f}, Accuracy={restored_val_accuracy:.4f}")
print("Der abschließende Test liefert eine separate Leistungsschätzung auf unbenutzten Daten.")

## 6. Model Evaluation auf Test Set

Jetzt testen wir das trainierte Modell auf dem echten Test Set.

In [ ]:
# Evaluation auf Test Set
test_loss, test_accuracy = model.evaluate(
    X_test_scaled, y_test_onehot, verbose=0
)

print("🎯 Test Set Performance:")
print(f"Test Loss: {test_loss:.4f}")
print(f"Test Accuracy: {test_accuracy:.4f} ({test_accuracy*100:.2f}%)")

# Vorhersagen machen
y_pred_proba = model.predict(X_test_scaled, verbose=0)
y_pred = np.argmax(y_pred_proba, axis=1)

# Classification Report
print("\n📊 Detaillierte Klassifikation:")
print(classification_report(y_test, y_pred, target_names=iris.target_names))

# Konfusionsmatrix
plt.figure(figsize=(8, 6))
cm = confusion_matrix(y_test, y_pred)
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues',
           xticklabels=iris.target_names, yticklabels=iris.target_names)
plt.title(f'Confusion Matrix\nTest Accuracy: {test_accuracy:.3f}', fontweight='bold')
plt.xlabel('Predicted')
plt.ylabel('Actual')
plt.show()

# Fehleranalyse
errors = np.where(y_test != y_pred)[0]
if len(errors) > 0:
    print(f"\n🔍 Fehleranalyse: {len(errors)} Fehler von {len(y_test)} Samples")
    for i in errors:
        actual = iris.target_names[y_test[i]]
        predicted = iris.target_names[y_pred[i]]
        confidence = np.max(y_pred_proba[i])
        print(f"Sample {i}: {actual} → {predicted} (Confidence: {confidence:.3f})")
else:
    print("\n🎉 Perfekte Klassifikation! Keine Fehler im Test Set.")

## 7. Modellwahrscheinlichkeiten für neue Blüten

Softmax zeigt, wie das Modell seine Wahrscheinlichkeit auf die Klassen verteilt.
Ein hoher Wert bedeutet eine starke Präferenz, nicht garantierte Richtigkeit.
Auch ein überzeugtes Modell kann falsch liegen.

Die Entropie fasst diese Verteilung zusammen: klein bei einer stark bevorzugten
Klasse, größer bei gleichmäßigerer Verteilung. Sie misst nicht, ob die Vorhersage
richtig ist. Dafür brauchen wir Beispiele mit bekannten richtigen Antworten.

In [ ]:
# Neue Beispiel-Blüten für Vorhersagen
new_flowers = np.array([
    [5.1, 3.5, 1.4, 0.2],  # Typische Setosa
    [6.9, 3.1, 4.9, 1.5],  # Typische Versicolor
    [6.3, 3.3, 6.0, 2.5],  # Typische Virginica
    [5.8, 2.7, 4.1, 1.0],  # Grenzfall
    [4.5, 2.0, 3.5, 1.0]   # Schwieriger Fall
])

# Skalierung anwenden
new_flowers_scaled = scaler.transform(new_flowers)

# Vorhersagen mit Keras
predictions_proba = model.predict(new_flowers_scaled, verbose=0)
predictions = np.argmax(predictions_proba, axis=1)

print("🔮 Keras Vorhersagen für neue Iris-Blüten:")
print("=" * 70)

feature_names = ['Sepal Length', 'Sepal Width', 'Petal Length', 'Petal Width']

for i, (flower, pred, probs) in enumerate(zip(new_flowers, predictions, predictions_proba)):
    print(f"\n🌸 Blüte {i+1}:")
    print(f"Features: {dict(zip(feature_names, flower))}")
    print(f"Vorhersage: {iris.target_names[pred]} (Klasse {pred})")

    print("Wahrscheinlichkeiten:")
    for j, (species, prob) in enumerate(zip(iris.target_names, probs)):
        bar = "█" * int(prob * 20)  # Visual bar
        print(f"  {species:10}: {prob:.4f} ({prob*100:5.1f}%) {bar}")

    # Confidence Analysis
    max_prob = np.max(probs)
    entropy = -np.sum(probs * np.log(probs + 1e-10))  # Uncertainty measure

    if max_prob > 0.95:
        confidence = "Stark auf eine Klasse konzentriert"
    elif max_prob > 0.8:
        confidence = "Überwiegend auf eine Klasse konzentriert"
    elif max_prob > 0.6:
        confidence = "Weniger eindeutig"
    else:
        confidence = "Auf mehrere Klassen verteilt"

    print(f"Modellausgabe: {confidence}")
    print(f"Max Probability: {max_prob:.4f}")
    print(f"Entropie der Modellwahrscheinlichkeiten: {entropy:.4f}")
    print("-" * 50)

In [ ]:
# Wahrscheinlichkeiten visualisieren
plt.figure(figsize=(15, 10))

for i in range(len(new_flowers)):
    plt.subplot(2, 3, i+1)

    # Balkendiagramm
    bars = plt.bar(iris.target_names, predictions_proba[i], 
                  color=['red', 'green', 'blue'], alpha=0.7)

    # Vorhersage hervorheben
    predicted_idx = predictions[i]
    bars[predicted_idx].set_alpha(1.0)
    bars[predicted_idx].set_edgecolor('black')
    bars[predicted_idx].set_linewidth(3)

    plt.title(f'Blüte {i+1}: {iris.target_names[predicted_idx]}', 
             fontweight='bold', fontsize=12)
    plt.ylabel('Wahrscheinlichkeit')
    plt.ylim(0, 1)

    # Werte anzeigen
    for bar, prob in zip(bars, predictions_proba[i]):
        plt.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.02,
                f'{prob:.3f}', ha='center', va='bottom', fontweight='bold')

    # Entropy als Unsicherheitsmaß
    entropy = -np.sum(predictions_proba[i] * np.log(predictions_proba[i] + 1e-10))
    plt.text(0.5, 0.85, f'Entropy: {entropy:.3f}', transform=plt.gca().transAxes,
            ha='center', bbox=dict(boxstyle='round', facecolor='wheat', alpha=0.8))

plt.tight_layout()
plt.show()

# Model-Interpretabilität
print("\n🧠 Model-Insights:")
print(f"Parameter insgesamt: {model.count_params():,}, davon trainierbar: {trainable_params:,}")

# Architektur korrekt extrahieren
architecture = []
for layer in model.layers:
    if hasattr(layer, 'units'):  # Dense layers
        architecture.append(layer.units)
print(f"🏗️  Architektur: {architecture}")

print(f"⚙️  Optimizer: Adam")
print(f"🎯 Best Val Accuracy: {max(val_acc):.4f}")
print(f"🔥 Final Test Accuracy: {test_accuracy:.4f}")

first_dense_layer = next(layer for layer in model.layers if isinstance(layer, layers.Dense))
first_layer_weights = first_dense_layer.get_weights()[0]
print(f"Gewichtsmatrix der ersten Dense-Schicht: {first_layer_weights.shape}")
print("Die Größe einzelner Gewichte zeigt nicht zuverlässig, wie wichtig ein Merkmal ist.")
print("Die Vorhersage hängt vom Zusammenspiel aller Schichten ab.")

## 8. Modell und Scaler speichern und laden

Wir speichern das Modell im aktuellen Keras-Format `.keras` und den Scaler separat.
Neue Daten müssen genauso skaliert werden wie beim Training.
Das ist ein einfaches Wiederverwendungsbeispiel, kein vollständiges Produktionssystem.
Lade Pickle-Dateien nur aus vertrauenswürdiger Quelle, zum Beispiel die hier selbst erzeugte Datei.

In [ ]:
# Model und Scaler speichern
model_path = "iris_keras_model.keras"
scaler_path = "iris_scaler.pkl"

# Keras Model speichern
model.save(model_path)
print(f"✅ Model gespeichert: {model_path}")

# Scaler mit pickle speichern
import pickle
with open(scaler_path, 'wb') as f:
    pickle.dump(scaler, f)
print(f"✅ Scaler gespeichert: {scaler_path}")

# Model laden und testen (Beispiel)
print("\n🔄 Model-Loading Test:")
loaded_model = keras.models.load_model(model_path, compile=False)
with open(scaler_path, 'rb') as f:
    loaded_scaler = pickle.load(f)

# Test mit geladenem Model
test_sample = np.array([[5.0, 3.0, 1.5, 0.3]])
test_sample_scaled = loaded_scaler.transform(test_sample)
prediction = loaded_model.predict(test_sample_scaled, verbose=0)

print(f"Test-Vorhersage: {iris.target_names[np.argmax(prediction)]}")
print(f"Wahrscheinlichkeiten: {prediction[0]}")
print("✅ Model erfolgreich geladen und getestet!")

# Model-Export Informationen
print(f"\n📦 Export-Informationen:")
print(f"Model-Datei: {model_path}")
print(f"Scaler-Datei: {scaler_path}")
print(f"TensorFlow Version: {tf.__version__}")
print(f"Model Input Shape: {model.input_shape}")
print(f"Model Output Shape: {model.output_shape}")

# Beispiel zum Laden und Vorhersagen
print(f"\n💻 Beispiel-Code:")
print("""
# Laden und Verwenden des Modells:
import tensorflow as tf
import pickle
import numpy as np

# Model und Scaler laden
model = tf.keras.models.load_model('iris_keras_model.keras', compile=False)
with open('iris_scaler.pkl', 'rb') as f:
    scaler = pickle.load(f)

# Vorhersage für neue Daten
def predict_iris(sepal_length, sepal_width, petal_length, petal_width):
    features = np.array([[sepal_length, sepal_width, petal_length, petal_width]])
    features_scaled = scaler.transform(features)
    prediction = model.predict(features_scaled)
    species = ['setosa', 'versicolor', 'virginica']
    return species[np.argmax(prediction)], prediction[0]

# Beispiel-Verwendung
species, probabilities = predict_iris(5.1, 3.5, 1.4, 0.2)
print(f"Vorhersage: {species}")
print(f"Wahrscheinlichkeiten: {probabilities}")
""")

## Zusammenfassung: Keras und scikit-learn

| Thema | scikit-learn MLPClassifier | TensorFlow/Keras |
|-------|---------------------------|------------------|
| Training beobachten | Trainingsfehler; bei Early Stopping auch Validierungs-Accuracy | Trainings- und Validierungsfehler sowie Accuracy |
| Early Stopping | Mit `early_stopping=True` möglich | Über einen Callback konfigurierbar |
| Klassenwahrscheinlichkeiten | `predict_proba` | Softmax-Ausgabe mit `predict` |
| Architektur | Gut für einfache vollständig verbundene Netze | Zusätzliche Schichttypen, Dropout und Batch Normalization |
| Speichern | Zum Beispiel mit Pickle oder Joblib | Zum Beispiel im `.keras`-Format |
| GPU | MLPClassifier nutzt die CPU | Möglich bei passender Hardware und Installation |

**Für dieses Beispiel gelernt:**
- Training, Validierung und Test haben unterschiedliche Aufgaben.
- Der Scaler darf nur aus den eigentlichen Trainingsdaten lernen.
- Early Stopping reagiert auf ausbleibende Verbesserung, nicht auf einen bewiesenen Overfitting-Fall.
- Lernkurven helfen beim Beobachten; der Test liefert eine separate Leistungsschätzung.
- Modellwahrscheinlichkeiten und Entropie sind keine Garantie für richtige Vorhersagen.
- Große Gewichte allein sind keine verlässliche Feature Importance.

Für einfache tabellarische Aufgaben ist scikit-learn oft ein guter Einstieg.
Keras bietet mehr Möglichkeiten für komplexe Netzarchitekturen. Es gibt auch andere
Deep-Learning-Werkzeuge, zum Beispiel PyTorch. Die Wahl hängt von der Aufgabe ab,
nicht von einer festen Grenze für die Anzahl der Datenpunkte.